# 06 — ETL FINAL: corpus amplio + filtros estructurales

**Este es el notebook definitivo de extracción.** Combina:

1. **Queries amplias temáticas (v2):** maximizan volumen del corpus.
2. **Filtros estructurales pandas (v4):** quitan ruido sistemático (recovery scammers, blog promos, metáfora institucional, política, marcas...).
3. **NO intenta separar "fraude real" de "discusión sobre fraude"** — eso es trabajo del clasificador zero-shot (BART-MNLI) en el siguiente notebook.

## Justificación metodológica

Tu pipeline de TFM contempla una clasificación posterior con BART-MNLI que tiene una etiqueta `"no relacionado con fraude financiero"`. Esa fase es la que separa los matices semánticos (¿metáfora o descripción? ¿víctima o testigo?). Los regex pandas no pueden hacer ese trabajo bien.

Esta separación es metodológicamente más limpia:
- **Extracción:** captura amplia con palabras clave (esta fase).
- **Limpieza estructural:** filtros deterministas para ruido evidente (esta fase).
- **Clasificación semántica:** BART-MNLI separa relevancia y tipología (próximo notebook).

## Configuración

- `PAGES_PER_QUERY = 8` (tirada definitiva).
- 5 queries × 8 páginas × 500 = hasta 20.000 tweets brutos.
- Coste API: **hasta 40 llamadas**.


In [ ]:
# Imports y carga del token
import os
import time
import requests
import pandas as pd
import re
from dotenv import load_dotenv

pd.set_option('display.max_colwidth', None)

load_dotenv()
TOKEN = os.getenv("X_BEARER_TOKEN", "")
if not TOKEN:
    raise ValueError("Falta X_BEARER_TOKEN en .env")

BASE_URL = "https://api.x.com/2/tweets/search/all"


In [ ]:
def search_all_posts_to_df(
    bearer_token, query, max_pages=8, max_results=500,
    sleep_seconds=1.1, start_time=None, end_time=None, label=""
):
    headers = {"Authorization": f"Bearer {bearer_token}"}
    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": "id,text,created_at,author_id,lang,geo,public_metrics,entities",
        "expansions": "author_id,geo.place_id",
        "user.fields": "id,name,username,created_at,location,verified,public_metrics",
        "place.fields": "id,full_name,country,country_code,place_type,geo",
        "sort_order": "recency",
    }
    if start_time: params["start_time"] = start_time
    if end_time: params["end_time"] = end_time

    all_rows = []
    next_token = None

    for page in range(max_pages):
        if next_token:
            params["next_token"] = next_token
        else:
            params.pop("next_token", None)

        retries = 0
        while True:
            r = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = min(60 * (2 ** retries), 900)
                print(f"  Rate limit. Esperando {wait}s...")
                time.sleep(wait)
                retries += 1
                if retries > 5: raise RuntimeError("Demasiados reintentos")
            elif r.status_code == 400:
                raise RuntimeError(f"400 Bad Request. Query len={len(query)}. {r.text}")
            elif r.status_code == 403:
                raise RuntimeError(f"403 Forbidden. {r.text}")
            else:
                raise RuntimeError(f"X API error {r.status_code}: {r.text}")

        payload = r.json()
        data = payload.get("data", []) or []
        includes = payload.get("includes", {}) or {}
        meta = payload.get("meta", {}) or {}
        users_by_id = {u["id"]: u for u in includes.get("users", []) or []}
        places_by_id = {p["id"]: p for p in includes.get("places", []) or []}

        for t in data:
            author = users_by_id.get(t.get("author_id"), {})
            place_id = (t.get("geo") or {}).get("place_id")
            place = places_by_id.get(place_id, {}) if place_id else {}
            entities = t.get("entities") or {}
            metrics = t.get("public_metrics") or {}
            am = author.get("public_metrics") or {}

            all_rows.append({
                "tweet_id": t.get("id"),
                "created_at": t.get("created_at"),
                "text": t.get("text"),
                "lang": t.get("lang"),
                "author_id": t.get("author_id"),
                "username": author.get("username"),
                "name": author.get("name"),
                "user_location": author.get("location"),
                "user_verified": author.get("verified"),
                "user_followers": am.get("followers_count"),
                "user_tweet_count": am.get("tweet_count"),
                "place_id": place_id,
                "place_full_name": place.get("full_name"),
                "place_country": place.get("country"),
                "place_country_code": place.get("country_code"),
                "place_type": place.get("place_type"),
                "retweet_count": metrics.get("retweet_count"),
                "reply_count": metrics.get("reply_count"),
                "like_count": metrics.get("like_count"),
                "quote_count": metrics.get("quote_count"),
                "n_hashtags": len(entities.get("hashtags", []) or []),
                "n_mentions": len(entities.get("mentions", []) or []),
                "n_urls": len(entities.get("urls", []) or []),
                "query_label": label,
            })

        print(f"  [{label} | p{page+1}/{max_pages}] {len(all_rows)} acumulados")
        next_token = meta.get("next_token")
        if not next_token:
            print(f"  [{label}] sin más páginas")
            break
        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)


## Queries amplias (las del v2 que demostraron volumen)

Estas son las queries originales del notebook 02 que retornaron 2.500 tweets en la tirada diagnóstica. Las mantenemos íntegras: el filtrado fino lo hará BART-MNLI después.


In [ ]:
EX_MIN = "-trump -biden -election -voter -ballot -nba -nfl -kpop"
GEO_OPS = "lang:en place_country:US -is:retweet -is:reply -is:nullcast"

def build(positive):
    q = f"({positive}) {EX_MIN} {GEO_OPS}"
    if len(q) > 512:
        raise ValueError(f"Query excede 512 chars: {len(q)}")
    return q

QUERIES = {
    "phishing": build(
        '"phishing email" OR "phishing text" OR "scam text" '
        'OR "scam email" OR "smishing" OR "got a text from" OR "fake text"'
    ),
    "payment_apps": build(
        '"Zelle scam" OR "Venmo scam" OR "Cash App scam" '
        'OR "PayPal scam" OR "Apple Pay scam"'
    ),
    "crypto": build(
        '"rug pull" OR "pig butchering" OR "crypto scam" '
        'OR "bitcoin scam" OR "investment scam" OR "ponzi scheme"'
    ),
    "romance_monetary": build(
        '("romance scam" OR "dating scam") '
        '(money OR sent OR wire OR transferred)'
    ),
    "impersonation": build(
        '"IRS scam" OR "USPS scam" OR "FedEx scam" OR "toll scam" '
        'OR "fake delivery" OR "fake invoice" '
        'OR "tech support scam" OR "job scam"'
    ),
}

print("=== QUERIES FINALES ===\n")
for label, q in QUERIES.items():
    print(f"{label:<20} {len(q):>4} chars")
    print(f"  {q}\n")


## Confirmación de la tirada definitiva

⚠️ **ESTA CELDA LANZA LA TIRADA DEFINITIVA.** Léela bien antes de pulsar Enter.


In [ ]:
PAGES_PER_QUERY = 8     # Definitiva
START = "2025-01-01T00:00:00Z"
END   = "2025-12-31T23:59:59Z"

est_calls = len(QUERIES) * PAGES_PER_QUERY
est_max_tweets = est_calls * 500
print("=" * 60)
print("   TIRADA DEFINITIVA — VERIFICACIÓN FINAL")
print("=" * 60)
print(f"  Páginas por query:        {PAGES_PER_QUERY}")
print(f"  Categorías:               {len(QUERIES)}")
print(f"  Llamadas API:             hasta {est_calls}")
print(f"  Tweets brutos máx:        {est_max_tweets}")
print(f"  Ventana temporal:         {START} → {END}")
print(f"  Tiempo estimado:          ~8-10 minutos")
print("=" * 60)
print()
print("⚠️  Esto consume tu cuota API. Confirma SOLO si:")
print("    1. El token en .env es correcto.")
print("    2. La ventana temporal es la que quieres.")
print("    3. Estás conforme con las queries de arriba.")
print()
confirmation = input("Escribe LANZAR para ejecutar, cualquier otra cosa para abortar: ")
if confirmation.strip().upper() != "LANZAR":
    raise RuntimeError("Tirada abortada por el usuario.")
print("\n✓ Confirmación recibida. Iniciando...\n")


## Ejecución

In [ ]:
dfs = []
for label, query in QUERIES.items():
    print(f"\n>>> {label}")
    df_part = search_all_posts_to_df(
        bearer_token=TOKEN, query=query,
        max_pages=PAGES_PER_QUERY, max_results=500,
        sleep_seconds=1.1, start_time=START, end_time=END,
        label=label,
    )
    print(f">>> {label}: {len(df_part)}")
    dfs.append(df_part)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\n=== TOTAL BRUTO: {len(df_raw)} ===")
print("\nPor categoría:")
print(df_raw["query_label"].value_counts())


## Guardado inmediato del bruto

⚠️ **Antes de aplicar filtros guardamos el corpus bruto.** Si algo va mal con los filtros, no tienes que volver a llamar a la API.


In [ ]:
os.makedirs("../data/raw", exist_ok=True)
df_raw.to_csv("../data/raw/scam_us_FINAL_raw.csv", index=False, encoding="utf-8")
print(f"✓ Guardado bruto: ../data/raw/scam_us_FINAL_raw.csv ({len(df_raw)} filas)")
print()
print("🚨 HAZ COPIA DE SEGURIDAD INMEDIATA EN DRIVE/DROPBOX")
print("    Este CSV NO se sube a GitHub (está en .gitignore).")
print("    Si lo pierdes, has perdido la tirada API.")


## Deduplicación entre queries

In [ ]:
df_raw["query_labels"] = df_raw.groupby("tweet_id")["query_label"].transform(
    lambda s: ",".join(sorted(set(s)))
)
df_dedup = df_raw.drop_duplicates(subset="tweet_id", keep="first").reset_index(drop=True)
df_dedup = df_dedup.drop(columns=["query_label"])
df_dedup.to_csv("../data/raw/scam_us_FINAL_dedup.csv", index=False, encoding="utf-8")
print(f"Únicos tras deduplicar: {len(df_dedup)}")
print(f"Multi-categoría: {(df_dedup['query_labels'].str.contains(',')).sum()}")
print(f"\n✓ Guardado deduplicado: ../data/raw/scam_us_FINAL_dedup.csv")


## Filtros estructurales (los del v4)

Quitan ruido sistemático que no requiere comprensión semántica:
- Tweets cortos, spam de keywords/menciones, no-EEUU.
- Recovery scammers (patrones DM + recuperar fondos).
- Blog promos / titulares de prensa.
- Metáfora institucional ("Social Security is a Ponzi", etc.).
- Frases hechas (life is a scam, etc.).
- Cuentas de marca conocidas (CNN, LifeLock, etc.).
- Política conspirativa, catfishing emocional puro.

⚠️ **NO intenta distinguir "fraude real" de "comentario sobre fraude"** — eso es trabajo del zero-shot.


In [ ]:
STRONG_FRAUD_TERMS = re.compile(
    r"\b(scam|scammed|scammer|scammers|fraud|fraudster|defrauded|"
    r"phishing|smishing|vishing|ponzi|rug\s*pull|pig\s*butchering|"
    r"identity\s*theft|wire\s*fraud|impersonat|spoof|"
    r"fake\s*(?:text|email|delivery|invoice|website|caller))\b",
    re.IGNORECASE,
)
MONEY_TERMS = re.compile(
    r"(\b(money|cash|dollar|dollars|usd|paid|payment|sent|wired|wire|"
    r"transfer|refund|bank|account|credit\s*card|debit|wallet|invoice|"
    r"charged|stolen|lost|victim|defrauded|deposit|withdraw|"
    r"check|venmo|zelle|paypal|crypto|bitcoin|coinbase|gift\s*card)\b|"
    r"\$\s*\d+|\d+\s*(k|dollars|usd))",
    re.IGNORECASE,
)
RECOVERY_SCAM_PATTERNS = re.compile(
    r"\b(DM\s+(now|me|us|asap)|we\s+are\s+tracing|get\s+your\s+money\s+back|"
    r"recover(y|\s+expert|\s+specialist)|funds\s+recovery|crypto\s+recovery|"
    r"contact\s+us\s+(?:now|asap|today)|reach\s+out\s+to\s+(?:us|me)|"
    r"100%\s+(?:guaranteed|recovery|success))\b",
    re.IGNORECASE,
)
BRAND_LIST_PATTERN = re.compile(
    r"(nft\s+scam|bitcoin\s+scam|forex|cfd|binary\s+option|"
    r"wallet\s+drained|bored\s*apes?|ethereum\s+scam)",
    re.IGNORECASE,
)
def is_recovery_scammer(text):
    if not isinstance(text, str): return False
    if RECOVERY_SCAM_PATTERNS.search(text): return True
    has_contact = bool(re.search(r"\b(DM|dm)\b|contact|reach\s+out", text, re.IGNORECASE))
    return has_contact and len(BRAND_LIST_PATTERN.findall(text)) >= 2

PROMO_BLOG_PATTERN = re.compile(
    r"\b(our\s+(?:recent\s+)?(?:blog|article|post)|breaks?\s+down|"
    r"read\s+(?:more|the\s+full)|link\s+in\s+bio|"
    r"check\s+(?:out|it)\s+(?:our|this)|"
    r"new\s+(?:blog|article|episode|video))\b",
    re.IGNORECASE,
)
NEWS_HEADLINE_PATTERN = re.compile(
    r"^\W*(leaders?|chief|ceo|president|founder|judge|jury|court|"
    r"police|sentence[d]?|charged|arrested|indicted|"
    r"\d+\s+(?:years?|months?)\s+in\s+prison)\b",
    re.IGNORECASE,
)
def looks_like_news_or_promo(text, n_urls=0):
    if not isinstance(text, str): return False
    if PROMO_BLOG_PATTERN.search(text): return True
    if n_urls > 0 and NEWS_HEADLINE_PATTERN.search(text): return True
    return False

INSTITUTIONAL_TARGETS = [
    "social security", "stock market", "housing market", "real estate market",
    r"\bai\b", "artificial intelligence", "crypto industry",
    "the tax system", "tax system", "taxes", "capitalism", "the government",
    "modern slavery", "slave engine", "marriage", "religion", "the church",
    "college", "the system", "the economy", "the federal reserve",
    "fiat currency", "social welfare", "welfare system",
    "healthcare system", "education system", "school system",
    "pension", "401k", "wall street",
]
ACCUSATION_WORDS = ["ponzi", "scheme", "scam", "fraud"]
WINDOW = r"(?:\W+\w+){0,6}\W+"
INSTITUTIONAL_METAPHOR_RE = re.compile(
    "|".join(
        f"({inst}{WINDOW}(?:{'|'.join(ACCUSATION_WORDS)}))"
        f"|((?:{'|'.join(ACCUSATION_WORDS)}){WINDOW}{inst})"
        for inst in INSTITUTIONAL_TARGETS
    ),
    re.IGNORECASE,
)
def is_institutional_metaphor(text):
    if not isinstance(text, str): return False
    return bool(INSTITUTIONAL_METAPHOR_RE.search(text))

METAPHOR_PATTERNS = re.compile(
    r"\b(life is a scam|love is a scam|system is a scam|"
    r"capitalism is a scam|college is a scam|dating is a scam|"
    r"my life is a scam|everything is a scam)\b",
    re.IGNORECASE,
)
SOFT_POLITICS_PATTERNS = re.compile(
    r"\b(harris|desantis|newsom|kamala|obama|congress|senate|senator|"
    r"republican|democrat|gop|maga|lawmaker|politician|"
    r"voter fraud|election fraud|stolen election|rigged election|"
    r"deep state)\b",
    re.IGNORECASE,
)
BRAND_USERNAMES = {
    "lifelock", "aura", "norton", "nortononline", "mcafee",
    "cnn", "foxnews", "msnbc", "nytimes", "wsj", "reuters",
    "ap", "bbcworld", "abc", "abcnews", "nbcnews", "cbsnews",
}
EMOTIONAL_ONLY = re.compile(
    r"\b(catfish|catfished|catfishing|emotional scam|"
    r"scammed my heart|broke my heart)\b",
    re.IGNORECASE,
)

URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")
def clean_for_length(text):
    if not isinstance(text, str): return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

us_keywords = ["united states", "usa", "u.s.", " us ", " us,", "america",
    "new york", "california", "texas", "florida", "chicago",
    "los angeles", "boston", "seattle", "miami", "atlanta",
    "dallas", "houston", "philadelphia", "phoenix", "denver",
    "washington", "san francisco", "san diego"]
def looks_us(row):
    if row.get("place_country_code") == "US": return True
    loc = " " + str(row.get("user_location") or "").lower() + " "
    return any(k in loc for k in us_keywords)


In [ ]:
# Aplicar filtros
df = df_dedup.copy()
df["clean_text"]       = df["text"].apply(clean_for_length)
df["clean_len"]        = df["clean_text"].str.len()
df["n_words"]          = df["clean_text"].str.split().str.len().fillna(0).astype(int)
df["hashtag_ratio"]    = (df["n_hashtags"] / df["n_words"].replace(0, 1)).fillna(0)
df["mention_ratio"]    = (df["n_mentions"] / df["n_words"].replace(0, 1)).fillna(0)
df["is_metaphor"]      = df["text"].fillna("").apply(lambda t: bool(METAPHOR_PATTERNS.search(t)))
df["is_soft_politics"] = df["text"].fillna("").apply(lambda t: bool(SOFT_POLITICS_PATTERNS.search(t)))
df["is_emotional"]     = df["text"].fillna("").apply(lambda t: bool(EMOTIONAL_ONLY.search(t)))
df["is_brand_account"] = df["username"].fillna("").str.lower().isin(BRAND_USERNAMES)
df["likely_us"]        = df.apply(looks_us, axis=1)
df["has_strong_fraud"] = df["text"].fillna("").apply(lambda t: bool(STRONG_FRAUD_TERMS.search(t)))
df["has_money"]        = df["text"].fillna("").apply(lambda t: bool(MONEY_TERMS.search(t)))
df["is_recovery_scammer"] = df["text"].fillna("").apply(is_recovery_scammer)
df["is_news_or_promo"]    = df.apply(lambda r: looks_like_news_or_promo(r["text"], r.get("n_urls", 0)), axis=1)
df["is_institutional_metaphor"] = df["text"].fillna("").apply(is_institutional_metaphor)
df["semantically_relevant"] = df["has_strong_fraud"] | df["has_money"]

mask = (
    (df["clean_len"] >= 40) &
    (df["semantically_relevant"]) &
    (df["likely_us"]) &
    (df["hashtag_ratio"] < 0.3) &
    (df["mention_ratio"] < 0.4) &
    (~df["is_metaphor"]) &
    (~df["is_soft_politics"]) &
    (~df["is_emotional"]) &
    (~df["is_brand_account"]) &
    (~df["is_recovery_scammer"]) &
    (~df["is_news_or_promo"]) &
    (~df["is_institutional_metaphor"])
)
df_clean = df[mask].reset_index(drop=True)

print("=" * 60)
print("   RESUMEN FINAL DEL CORPUS")
print("=" * 60)
print(f"  Brutos descargados:        {len(df_raw)}")
print(f"  Únicos tras deduplicar:    {len(df_dedup)}")
print(f"  Limpios estructurales:     {len(df_clean)}")
print(f"  Retención:                 {len(df_clean)/max(len(df_dedup),1)*100:.1f}%")
print("=" * 60)
print("\nPor categoría temática:")
for label in QUERIES.keys():
    n = df_clean["query_labels"].fillna("").str.contains(label).sum()
    print(f"  {label:<20} {n:>6}")
print("\nDistribución de descartes:")
print(f"  Metáfora institucional:    {df['is_institutional_metaphor'].sum()}")
print(f"  Recovery scammers:         {df['is_recovery_scammer'].sum()}")
print(f"  Noticias / blog promos:    {df['is_news_or_promo'].sum()}")
print(f"  Política suave:            {df['is_soft_politics'].sum()}")
print(f"  Sin relevancia semántica:  {(~df['semantically_relevant']).sum()}")
print(f"  No-EEUU:                   {(~df['likely_us']).sum()}")


## Guardado final

In [ ]:
df_clean.to_csv("../data/raw/scam_us_FINAL_clean.csv", index=False, encoding="utf-8")
print(f"✓ Guardado limpio: ../data/raw/scam_us_FINAL_clean.csv ({len(df_clean)} filas)")
print()
print("📦 Archivos guardados:")
print(f"   1. scam_us_FINAL_raw.csv      ({len(df_raw)} filas)   ← bruto, antes de cualquier filtro")
print(f"   2. scam_us_FINAL_dedup.csv    ({len(df_dedup)} filas)   ← tras deduplicar")
print(f"   3. scam_us_FINAL_clean.csv    ({len(df_clean)} filas)   ← tras filtros estructurales")
print()
print("📌 PRÓXIMO PASO: notebook 07 con BERTopic + clasificador zero-shot BART-MNLI.")
print("   Ese notebook hace el filtrado fino (\"fraude real\" vs \"comentario sobre fraude\").")


## Inspección visual del corpus final

In [ ]:
print("=== MUESTRA ALEATORIA DE 20 TWEETS LIMPIOS ===\n")
for _, row in df_clean.sample(min(20, len(df_clean)), random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
    print(f"  {row['text'][:280]}")
    print()
